In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex AI Rag: Cross-Corpus Retrieval with `AskContexts` and `AsyncRetrieveContexts` Demo

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Frag-engine%2Fcross_corpus_retrieval.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/rag-engine/cross_corpus_retrieval.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/cross_corpus_retrieval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>            

| Author |
| --- |
| [Tiger Jin](https://github.com/tiger-w-jin) |



This notebook demonstrates how to invoke the new Vertex AI RAG handlers (`AskContexts` and `AsyncRetrieveContexts`) using the Vertex AI Python SDK for cross-corpus retrieval.

**Note**: Since these handlers are currently in preview states, this notebook relies on the `vertexai.preview.rag` module.

## Cross-Corpus Retrieval

Cross-corpus retrieval in Vertex AI RAG allows you to query multiple RAG corpora simultaneously with a single request. This is particularly powerful when your knowledge base is distributed across different datasets or when you need to retrieve information that might span across various domains or document types.

For more information, refer to the [official documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/cross-corpus-retrieval).

### `AskContexts` and `AsyncRetrieveContexts`

These are two key functions within the Vertex AI RAG framework, available through the `vertexai.preview.rag` module. They are designed to facilitate the retrieval of relevant contexts from configured RAG corpora.

*   **`AskContexts`**: This function is used for synchronous retrieval of contexts. It takes a query and a list of `RagResource` objects (which define the RAG corpora to search) and returns the most relevant contexts based on the `rag_retrieval_config`.

*   **`AsyncRetrieveContexts`**: Similar to `AskContexts`, but designed for asynchronous retrieval. This is particularly useful when dealing with multiple corpora or when integration into an asynchronous application flow is required, allowing for non-blocking operations.

## Get started

### Install Vertex AI SDK and other required packages


In [ ]:
%pip install google-cloud-aiplatform

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the cell below to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

## Initialization


### Set Google Cloud project information and initialize Vertex AI SDK

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
from google.cloud import aiplatform
import vertexai
from vertexai.preview import rag
from google.cloud import aiplatform_v1beta1

SERVICE = "aiplatform.googleapis.com"  # @param {type:"string"}
ENDPOINT = "us-central1-aiplatform.googleapis.com"  # @param {type:"string"}


PROJECT_ID = "[your-project-id]"  # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}

aiplatform.init(project=PROJECT_ID, location=LOCATION,
    api_endpoint=ENDPOINT
)
print(f"Vertex AI initialized with project: {PROJECT_ID}, location: {LOCATION}")
print("Available RAG functions exposed:")
print(f" - async_retrieve_contexts: {hasattr(rag, 'async_retrieve_contexts')}")
print(f" - ask_contexts: {hasattr(rag, 'ask_contexts')}")

## Rag Corpus management


### Option 1: Create a RAG Corpus

To create a new RAG corpus, use the `create_corpus` function from the `rag` module. This initializes a managed index for your documents. You can reference the [introductory notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/rag-engine/intro_rag_engine.ipynb) for more details.

### Option 2: Use Existing Rag Corpus Path


In [ ]:
CORPUS_PATH_1 = "[your-corpus-path]"  # @param {type:"string"}
CORPUS_PATH_2 = "[your-corpus-path]"  # @param {type:"string"}
print(f"Corpus 1 set to: {CORPUS_PATH_1}")
print(f"Corpus 2 set to: {CORPUS_PATH_2}")

rag_resources_list = [
    rag.RagResource(rag_corpus=CORPUS_PATH_1),
    rag.RagResource(rag_corpus=CORPUS_PATH_2),
]

## Rag Retrieval


### AskContexts


In [ ]:
# # Config
config = rag.RagRetrievalConfig(top_k=2)

query="[Your query]"# @param {type:"string"}

response = rag.ask_contexts(
    text=query,
    rag_resources=rag_resources_list,
    rag_retrieval_config=config,
)

print("Response:", response)

### AsyncRetrieveContexts


In [ ]:
query="[Your query]"# @param {type:"string"}

# Async retrieve contexts from multiple corpora
response = await rag.async_retrieve_contexts(
    text=query,
    rag_resources=rag_resources_list,
    rag_retrieval_config=config,
)
print(response)